In [2]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.arduino import Arduino_IO

base = BaseOverlay("base.bit")

pin_out = Arduino_IO(base.ARDUINO, 0, 'out')
pin_in  = Arduino_IO(base.ARDUINO, 1, 'in')

pin_out.write(1)
value = pin_in.read()
print(value)


1


In [3]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.arduino import Grove_OLED, ARDUINO_GROVE_I2C

print("Loading base overlay...")

base = BaseOverlay("base.bit")

print("Attempting to detect OLED on Arduino I2C...")

try:
    oled = Grove_OLED(base.ARDUINO, ARDUINO_GROVE_I2C)
    oled.clear()
    oled.write("OLED Found")
    
    print("SUCCESS: OLED detected and responding.")

except Exception as e:
    print("OLED NOT detected.")
    print("Error:", e)


Loading base overlay...
Attempting to detect OLED on Arduino I2C...
SUCCESS: OLED detected and responding.


In [5]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.arduino import Grove_OLED, ARDUINO_GROVE_I2C
import time

base = BaseOverlay("base.bit")
oled = Grove_OLED(base.ARDUINO, ARDUINO_GROVE_I2C)

oled.clear()
time.sleep(1)

oled.set_normal_mode()
oled.set_position(0, 0)
oled.write("I2C OK")
time.sleep(2)

oled.clear()
oled.set_position(1, 0)
oled.write("PYNQ OLED")


In [ ]:
from pynq.overlays.base import BaseOverlay

print("Loading base overlay...")

base = BaseOverlay("base.bit")

print("Overlay loaded successfully")
from pynq.lib.arduino import Grove_OLED
from pynq.lib.arduino import ARDUINO_GROVE_I2C

print("Libraries imported")


realized soon after that likely my oled screen isnt supported for the grove_oled library; not seeing an output on the screen after sending the above code.

In [1]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
import time

base = BaseOverlay("base.bit")
lib = MicroblazeLibrary(base.ARDUINO, ['i2c'])


i2c = lib.i2c_open_device(0)# Open Arduino I2C device 0; SDA--> arduino SDA pin; SCL--> arduino SCL pin
OLED_ADDR = 0x3C   # most SSD1306 modules use 0x3C

def oled_cmd(*cmds):  #<-- send one or more commands over i2c
    # 0x00 = command control byte
    buf = bytearray([0x00] + list(cmds))
    i2c.write(OLED_ADDR, buf, len(buf))

def oled_data(data_bytes):  #<-- send pixel data to the display over i2c
    # 0x40 = data control byte
    if isinstance(data_bytes, list):
        data_bytes = bytes(data_bytes)
    buf = bytearray([0x40]) + data_bytes
    i2c.write(OLED_ADDR, buf, len(buf))

def oled_set_pos(page, col):  #<-- set cursor positioning
    oled_cmd(0xB0 + page)              # page address
    oled_cmd(0x00 + (col & 0x0F))      # lower column
    oled_cmd(0x10 + (col >> 4))        # higher column

def oled_clear():   #<-- clear entire display
    for page in range(8):              # 8 pages for 64-pixel height
        oled_set_pos(page, 0)
        oled_data([0x00] * 128)

def oled_init():  #<-- 128x64 pixel matrix; initialize for operation
    oled_cmd(
        0xAE,       # display OFF
        0xD5, 0x80, # clock divide
        0xA8, 0x3F, # multiplex = 64
        0xD3, 0x00, # display offset
        0x40,       # start line = 0
        0x8D, 0x14, # charge pump ON
        0x20, 0x00, # horizontal addressing mode
        0xA1,       # segment remap
        0xC8,       # COM scan dec
        0xDA, 0x12, # COM pins config for 128x64
        0x81, 0xCF, # contrast
        0xD9, 0xF1, # pre-charge
        0xDB, 0x40, # VCOM detect
        0xA4,       # resume RAM display
        0xA6,       # normal display
        0xAF        # display ON
    )
    time.sleep(0.1)
    oled_clear()


FONT = {  #<--  5x8 font for characters, numbers, other ascii
   " ": [0x00,0x00,0x00,0x00,0x00],
    "-": [0x08,0x08,0x08,0x08,0x08],
    ".": [0x00,0x60,0x60,0x00,0x00],
    ":": [0x00,0x36,0x36,0x00,0x00],
    "/": [0x20,0x10,0x08,0x04,0x02],

    "0": [0x3E,0x51,0x49,0x45,0x3E],
    "1": [0x00,0x42,0x7F,0x40,0x00],
    "2": [0x42,0x61,0x51,0x49,0x46],
    "3": [0x21,0x41,0x45,0x4B,0x31],
    "4": [0x18,0x14,0x12,0x7F,0x10],
    "5": [0x27,0x45,0x45,0x45,0x39],
    "6": [0x3C,0x4A,0x49,0x49,0x30],
    "7": [0x01,0x71,0x09,0x05,0x03],
    "8": [0x36,0x49,0x49,0x49,0x36],
    "9": [0x06,0x49,0x49,0x29,0x1E],

    "A": [0x7E,0x11,0x11,0x11,0x7E],
    "B": [0x7F,0x49,0x49,0x49,0x36],
    "C": [0x3E,0x41,0x41,0x41,0x22],
    "D": [0x7F,0x41,0x41,0x22,0x1C],
    "E": [0x7F,0x49,0x49,0x49,0x41],
    "F": [0x7F,0x09,0x09,0x09,0x01],
    "G": [0x3E,0x41,0x49,0x49,0x7A],
    "H": [0x7F,0x08,0x08,0x08,0x7F],
    "I": [0x00,0x41,0x7F,0x41,0x00],
    "J": [0x20,0x40,0x41,0x3F,0x01],
    "K": [0x7F,0x08,0x14,0x22,0x41],
    "L": [0x7F,0x40,0x40,0x40,0x40],
    "M": [0x7F,0x02,0x0C,0x02,0x7F],
    "N": [0x7F,0x04,0x08,0x10,0x7F],
    "O": [0x3E,0x41,0x41,0x41,0x3E],
    "P": [0x7F,0x09,0x09,0x09,0x06],
    "Q": [0x3E,0x41,0x51,0x21,0x5E],
    "R": [0x7F,0x09,0x19,0x29,0x46],
    "S": [0x46,0x49,0x49,0x49,0x31],
    "T": [0x01,0x01,0x7F,0x01,0x01],
    "U": [0x3F,0x40,0x40,0x40,0x3F],
    "V": [0x1F,0x20,0x40,0x20,0x1F],
    "W": [0x3F,0x40,0x38,0x40,0x3F],
    "X": [0x63,0x14,0x08,0x14,0x63],
    "Y": [0x07,0x08,0x70,0x08,0x07],
    "Z": [0x61,0x51,0x49,0x45,0x43],
}

def oled_text(page, col, text):  #<-- draw ascii text in the beginning of page/column
    oled_set_pos(page, col)
    out = []
    for ch in text:
        pattern = FONT.get(ch.upper(), FONT[" "])
        out.extend(pattern + [0x00])   # spacing column
    oled_data(out[:128-col])

def oled_show_env(temp_c, temp_f, mic_level, status):  #<-- displays environment values/ warning status on oled
    """
    Display environmental values and warning status on OLED.
    """
    oled_clear()
    oled_text(0, 0, f"T C:{temp_c:.1f}")
    oled_text(1, 0, f"T F:{temp_f:.1f}")
    oled_text(3, 0, f"MIC:{mic_level:.1f}")
    oled_text(6, 0, status[:20])


In [4]:
#<-- demo test functionality
try:
    oled_init()
    oled_text(0, 0, "Remote Baby Monitor")
    oled_text(2, 0, "PYNQ-Z2")
    print("SSD1306 initialized and text sent.")
except Exception as e:
    print("OLED communication failed:", e)

SSD1306 initialized and text sent.


used this link to guide thru setup for the oled ssd1306: https://discuss.pynq.io/t/interface-of-oled-ssd1306-with-pynq-z2/6975
found this github with the ascii to hex converted: https://github.com/greiman/SSD1306Ascii/blob/master/src/fonts/font5x7.h